## 5. Compare with a real nontrivial host run

Paper 033 drives the Paper 032 governor on a weighted three-cycle.
The initial edge assertions have nonzero irreducible defect; a host
`CUT` replaces claims and checks zero. This is a *different declared
board* from the one-edge adapter, and that `CUT` changes claims. It
does not prove that Alpha observations always admit an Core repair.


In [ ]:
import json
import os
import sys
from fractions import Fraction
from hashlib import sha256
from pathlib import Path


def find_repositories():
    candidates = [Path.cwd(), *Path.cwd().parents]
    extra = os.environ.get("GENESIS_REPOS_HOME")
    if extra:
        candidates.insert(0, Path(extra))
    for root in candidates:
        has_genesis = (root / "genesis" / "3_alpha_mathematical_language" / "Cargo.toml").is_file() or (root / "3_alpha_mathematical_language" / "Cargo.toml").is_file()
        has_symbol = (root / "the_symbol_game" / "education").is_dir() or (root / "education").is_dir()
        if has_genesis and has_symbol:
            g_root = root / "genesis" if (root / "genesis").is_dir() else root
            s_root = root / "the_symbol_game" if (root / "the_symbol_game").is_dir() else root
            return g_root, s_root
    raise RuntimeError(
        "Place genesis and the_symbol_game beside each other or set GENESIS_REPOS_HOME"
    )


GENESIS, SYMBOL = find_repositories()
REPOS_HOME = GENESIS.parent
ALPHA = Path(os.environ.get("GENESIS_ALPHA_ROOT", GENESIS / "3_alpha_mathematical_language"))
_PFP_PAPERS = (
    SYMBOL
    / "atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
)
PAPER33 = _PFP_PAPERS / "005_pfp_practitioners_field_manual" / "code"
PAPER32 = _PFP_PAPERS / "004_pfp_native_verification_and_evidence_custody" / "code"
import importlib.util


def _load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


_load("l1_symbol_game_governor", PAPER32 / "l1_symbol_game_governor.py")
run_to_terminal = _load(
    "symbol_game_toolkit", PAPER33 / "symbol_game_toolkit.py"
).run_to_terminal


def digest(path):
    return sha256(Path(path).read_bytes()).hexdigest()


source_id = {
    "alpha_front_end_sha256": digest(ALPHA / "src" / "lib.rs"),
    "paper032_governor_sha256": digest(PAPER32 / "l1_symbol_game_governor.py"),
    "paper033_toolkit_sha256": digest(PAPER33 / "symbol_game_toolkit.py"),
}
print(json.dumps(source_id, indent=2))


In [ ]:
cycle = run_to_terminal(
    ["a", "b", "c"],
    [("a", "b", 1), ("b", "c", 2), ("c", "a", 2)],
    {("a", "b"): 1, ("b", "c"): 1, ("c", "a"): 1},
)
before = [Fraction(row["phi1"]) for row in cycle["trace"] if row["move"] == "ASSERT"]
assert before == [1, 2, 3]
assert cycle["terminal"] == "ACCEPTED_ZERO_DEFECT"
assert Fraction(cycle["final_phi1"]) == 0
assert any(row["move"] == "CUT" for row in cycle["trace"])
for row in cycle["trace"]:
    print(row["move"], row["status"], "Phi_1=", row["phi1"])

> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `13_alpha_to_core_execution`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
